# 🤝 GRPO Negotiation Training – Single GPU Colab (Modular)

Uses [Franceschetti's multiturn-llm-training](https://github.com/lfranceschetti/multiturn-llm-training) repo directly for:
- `envs/negotiation/` → Game definitions, payoff tables, prompts, reward functions
- `evaluator/` → Outcome extraction via GPT-4o-mini

Only the **model loading** (Unsloth) and **training loop** (no vLLM) are custom.

**Setup:** Single A100, Unsloth 4-bit, LoRA adapters.


In [ ]:
# ============================================================
# Cell 1: Install Dependencies & Clone Repo
# ============================================================
!pip install -q unsloth
!pip install -q peft accelerate bitsandbytes
!pip install -q openai pyyaml wandb
!pip install -q attrs hydra-core omegaconf

# Clone Luca's repo
!git clone https://github.com/lfranceschetti/multiturn-llm-training.git
%cd /content/multiturn-llm-training

# Add repo to Python path so we can import from it
import sys
sys.path.insert(0, "/content/multiturn-llm-training")
sys.path.insert(0, "/content/multiturn-llm-training/envs/negotiation")

print("Repo cloned and paths configured.")


In [ ]:
# ============================================================
# Cell 2: Setup OpenAI Key & Configuration
# ============================================================
import os
import json
import torch
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- OpenAI API Key ---
# The evaluator module reads from secrets.json
OPENAI_KEY = ""  # ← Paste your OpenAI key here

os.environ["OPENAI_API_KEY"] = OPENAI_KEY

# Create secrets.json for Luca's evaluator module
secrets = {"openai": {"api_key": OPENAI_KEY}}
with open("/content/multiturn-llm-training/secrets.json", "w") as f:
    json.dump(secrets, f)

# --- Training Config ---
CONFIG = {
    "model_name": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "game_type": "generic-rental-agreement",
    "max_rounds": 5,
    "max_tokens_per_turn": 200,
    "group_size": 8,            # G dialogues per prompt
    "learning_rate": 5e-5,
    "beta": 0.08,               # KL penalty
    "num_train_steps": 200,
    "eval_every": 25,
    "save_every": 50,
    "temperature": 1.0,
    "eval_temperature": 0.7,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "num_eval_episodes": 20,
}

print("Config loaded.")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


In [ ]:
# ============================================================
# Cell 3: Load Model with Unsloth + LoRA
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {CONFIG['model_name']}")
model.print_trainable_parameters()


In [ ]:
# ============================================================
# Cell 4: Initialize Luca's Negotiation Environment
# ============================================================
# Import directly from the cloned repo
from envs.negotiation.env import NegotiationEnv
from envs.negotiation.games import Game

# Create the environment
env = NegotiationEnv(game_type=CONFIG["game_type"], seed=SEED)

# Get reward functions (uses GPT-4o-mini evaluator internally)
reward_functions = env.get_reward_functions()

# Create a dataset to get prompts and game configs
dataset = env.create_dataset(size=100)

print(f"Environment: {CONFIG['game_type']}")
print(f"Dataset size: {len(dataset)}")
print(f"Sample keys: {list(dataset[0].keys())}")
print(f"\nSample prompt (first 300 chars):")
print(dataset[0]["prompt"][:300])


In [ ]:
# ============================================================
# Cell 5: Multi-Turn Dialogue Generation (replaces vLLM)
# ============================================================
from typing import Tuple, List, Dict

@torch.no_grad()
def generate_response(model, tokenizer, messages: list,
                      max_new_tokens: int = 200, temperature: float = 1.0) -> str:
    \"\"\"Generate a single response from a conversation history.\"\"\"
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text, return_tensors="pt", truncation=True, max_length=1800
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=max(temperature, 0.01),
        do_sample=True,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def play_negotiation(model, tokenizer, prompt_agent: str, prompt_opponent: str,
                     agent_starts: bool = True, max_rounds: int = 5,
                     temperature: float = 1.0) -> Tuple[list, list]:
    \"\"\"
    Play a full multi-turn negotiation. Both roles use the same model.

    Returns:
        conversation: List of message dicts [{role, content}, ...]
        agent_turn_indices: Which messages in `conversation` are the agent's
    \"\"\"
    agent_history = [{"role": "system", "content": prompt_agent}]
    opponent_history = [{"role": "system", "content": prompt_opponent}]
    conversation = []
    agent_turn_indices = []

    for round_num in range(max_rounds):
        # Determine turn order
        if agent_starts:
            speakers = ["agent", "opponent"]
        else:
            speakers = ["opponent", "agent"]

        for speaker in speakers:
            if speaker == "agent":
                response = generate_response(
                    model, tokenizer, agent_history,
                    max_new_tokens=CONFIG["max_tokens_per_turn"],
                    temperature=temperature,
                )
                agent_history.append({"role": "assistant", "content": response})
                opponent_history.append({"role": "user", "content": response})
                agent_turn_indices.append(len(conversation))
                conversation.append({"role": "assistant", "content": response})
            else:
                response = generate_response(
                    model, tokenizer, opponent_history,
                    max_new_tokens=CONFIG["max_tokens_per_turn"],
                    temperature=temperature,
                )
                opponent_history.append({"role": "assistant", "content": response})
                agent_history.append({"role": "user", "content": response})
                conversation.append({"role": "user", "content": response})

    return conversation, agent_turn_indices


# Quick test
print("Testing dialogue generation...")
FastLanguageModel.for_inference(model)

sample = dataset[0]
conv, idx = play_negotiation(
    model, tokenizer, sample["prompt"], sample["prompt_2"],
    agent_starts=sample["starting_agent"],
    max_rounds=2, temperature=1.0,
)

print(f"Generated {len(conv)} messages, agent spoke at indices {idx}")
for i, msg in enumerate(conv):
    marker = " ← AGENT" if i in idx else ""
    role = "🏠 Agent" if msg["role"] == "assistant" else "🧑 Opponent"
    print(f"\n{role}{marker}: {msg['content'][:200]}")


In [ ]:
# ============================================================
# Cell 6: Reward Computation (uses Luca's evaluator)
# ============================================================

def compute_rewards(reward_functions, sample, conversations):
    \"\"\"
    Compute rewards for multiple conversations using Luca's reward functions.

    Args:
        reward_functions: From env.get_reward_functions()
        sample: A dataset sample with game_config, negotiation_role, etc.
        conversations: List of conversation message lists

    Returns:
        List of float rewards
    \"\"\"
    rewards = []
    for conv in conversations:
        # Luca's reward function expects lists of prompts/completions
        for reward_func in reward_functions:
            try:
                result = reward_func(
                    prompts=[sample["prompt"]],
                    completions=[conv],
                    get_full_info=False,
                    negotiation_roles=[sample.get("negotiation_role")],
                    game_configs=[sample.get("game_config")],
                )
                rewards.append(float(result[0]))
            except Exception as e:
                print(f"Reward computation error: {e}")
                rewards.append(0.0)
    return rewards


# Test with the dialogue from above
print("Testing reward computation...")
test_rewards = compute_rewards(reward_functions, sample, [conv])
print(f"Reward: {test_rewards}")


In [ ]:
# ============================================================
# Cell 7: Log-Probability Computation
# ============================================================

def compute_agent_logprobs(model, tokenizer, conversation, agent_turn_indices,
                           system_prompt, temperature=1.0):
    \"\"\"
    Compute log-probabilities for the agent's tokens only.

    Returns:
        total_log_prob: Sum of log-probs for agent tokens
        num_agent_tokens: Number of tokens the agent generated
    \"\"\"
    messages = [{"role": "system", "content": system_prompt}] + conversation

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    encoding = tokenizer(
        input_text, return_tensors="pt", truncation=True, max_length=2048
    )
    input_ids = encoding["input_ids"].to(model.device)
    attention_mask = encoding["attention_mask"].to(model.device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :] / max(temperature, 0.01)

    target_ids = input_ids[:, 1:]
    log_probs = torch.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)

    # Build assistant mask by finding token boundaries for agent messages
    assistant_mask = torch.zeros_like(target_ids, dtype=torch.float32)

    current_messages = [{"role": "system", "content": system_prompt}]
    for i, msg in enumerate(conversation):
        current_messages.append(msg)
        if i in agent_turn_indices:
            text_with = tokenizer.apply_chat_template(
                current_messages, tokenize=False, add_generation_prompt=False
            )
            text_without = tokenizer.apply_chat_template(
                current_messages[:-1], tokenize=False, add_generation_prompt=True
            )
            tokens_with = tokenizer(text_with, truncation=True, max_length=2048)["input_ids"]
            tokens_without = tokenizer(text_without, truncation=True, max_length=2048)["input_ids"]

            start = len(tokens_without) - 1
            end = len(tokens_with) - 1
            if start < assistant_mask.shape[1] and end <= assistant_mask.shape[1]:
                assistant_mask[0, start:end] = 1.0

    masked = token_log_probs * assistant_mask
    return masked.sum(), assistant_mask.sum().item()


# Test
print("Testing logprob computation...")
FastLanguageModel.for_training(model)
lp, nt = compute_agent_logprobs(model, tokenizer, conv, idx, sample["prompt"])
print(f"Log-prob: {lp.item():.2f}, Agent tokens: {nt:.0f}")


In [ ]:
# ============================================================
# Cell 8: GRPO Training Loop
# ============================================================
from torch.optim import AdamW
from unsloth import FastLanguageModel

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG["learning_rate"],
    weight_decay=0.01,
)

history = {"step": [], "mean_reward": [], "loss": [], "kl": []}

print("=" * 60)
print(f"Starting GRPO Training")
print(f"Steps: {CONFIG['num_train_steps']}, G={CONFIG['group_size']}, beta={CONFIG['beta']}")
print(f"Using Luca's NegotiationEnv for prompts & rewards")
print("=" * 60)

for step in range(1, CONFIG["num_train_steps"] + 1):

    # ---- PHASE 1: SAMPLE A SCENARIO ----
    sample_idx = random.randint(0, len(dataset) - 1)
    sample = dataset[sample_idx]

    # ---- PHASE 2: GENERATE G DIALOGUES ----
    FastLanguageModel.for_inference(model)

    conversations = []
    agent_indices_list = []

    for g in range(CONFIG["group_size"]):
        conv, agent_idx = play_negotiation(
            model, tokenizer,
            sample["prompt"], sample["prompt_2"],
            agent_starts=sample["starting_agent"],
            max_rounds=CONFIG["max_rounds"],
            temperature=CONFIG["temperature"],
        )
        conversations.append(conv)
        agent_indices_list.append(agent_idx)

    # ---- PHASE 3: COMPUTE REWARDS (via Luca's evaluator) ----
    rewards = compute_rewards(reward_functions, sample, conversations)
    rewards_tensor = torch.FloatTensor(rewards)

    # ---- PHASE 4: COMPUTE GROUP ADVANTAGES ----
    mean_r = rewards_tensor.mean()
    std_r = rewards_tensor.std()
    advantages = (rewards_tensor - mean_r) / (std_r + 1e-4)

    # ---- PHASE 5: COMPUTE LOSS & UPDATE ----
    FastLanguageModel.for_training(model)

    total_loss = torch.tensor(0.0, device=model.device, requires_grad=True)
    total_kl = 0.0
    n_valid = 0

    for g in range(CONFIG["group_size"]):
        adv = advantages[g].to(model.device)
        if abs(adv.item()) < 1e-6:
            continue

        # Current policy log-probs
        log_prob, n_tokens = compute_agent_logprobs(
            model, tokenizer, conversations[g], agent_indices_list[g],
            sample["prompt"], CONFIG["temperature"],
        )

        if n_tokens == 0:
            continue

        # Reference log-probs (LoRA disabled = base model)
        with torch.no_grad():
            model.disable_adapter_layers()
            ref_log_prob, _ = compute_agent_logprobs(
                model, tokenizer, conversations[g], agent_indices_list[g],
                sample["prompt"], CONFIG["temperature"],
            )
            model.enable_adapter_layers()

        # Per-token KL and loss
        kl = (log_prob - ref_log_prob) / max(n_tokens, 1)
        norm_log_prob = log_prob / max(n_tokens, 1)
        sample_loss = -norm_log_prob * adv + CONFIG["beta"] * kl

        total_loss = total_loss + sample_loss
        total_kl += kl.item()
        n_valid += 1

    if n_valid > 0:
        total_loss = total_loss / n_valid
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    # ---- LOGGING ----
    mean_reward = mean_r.item()
    loss_val = total_loss.item() if n_valid > 0 else 0.0
    kl_val = total_kl / max(n_valid, 1)

    history["step"].append(step)
    history["mean_reward"].append(mean_reward)
    history["loss"].append(loss_val)
    history["kl"].append(kl_val)

    if step % 5 == 0 or step == 1:
        rewards_str = [f"{r:.0f}" for r in rewards]
        print(f"Step {step:>4d} | Reward: {mean_reward:>6.1f} (±{std_r.item():.1f}) | "
              f"Loss: {loss_val:>7.4f} | KL: {kl_val:.4f} | "
              f"[{', '.join(rewards_str)}]")

    # ---- EVALUATION ----
    if step % CONFIG["eval_every"] == 0:
        print(f"\n--- Eval at step {step} ---")
        FastLanguageModel.for_inference(model)
        eval_rewards = []
        for _ in range(CONFIG["num_eval_episodes"]):
            s = dataset[random.randint(0, len(dataset) - 1)]
            ec, ei = play_negotiation(
                model, tokenizer, s["prompt"], s["prompt_2"],
                agent_starts=s["starting_agent"],
                max_rounds=CONFIG["max_rounds"],
                temperature=CONFIG["eval_temperature"],
            )
            er = compute_rewards(reward_functions, s, [ec])
            eval_rewards.append(er[0])
        print(f"Eval: {np.mean(eval_rewards):.1f} (±{np.std(eval_rewards):.1f}), "
              f"Agreement: {sum(1 for r in eval_rewards if r > 0)/len(eval_rewards)*100:.0f}%")
        print("---\n")

    # ---- SAVE ----
    if step % CONFIG["save_every"] == 0:
        path = f"/content/grpo_checkpoint_{step}"
        model.save_pretrained(path)
        tokenizer.save_pretrained(path)
        print(f"Saved: {path}")

print("\nTraining complete!")


In [ ]:
# ============================================================
# Cell 9: Plot Results
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, key, title, color in zip(
    axes,
    ["mean_reward", "loss", "kl"],
    ["Mean Group Reward", "GRPO Loss", "KL from Reference"],
    ["tab:blue", "tab:red", "tab:green"],
):
    vals = history[key]
    ax.plot(history["step"], vals, alpha=0.3, color=color)
    # Smoothed
    w = min(20, len(vals))
    if w > 1:
        smoothed = np.convolve(vals, np.ones(w)/w, mode="valid")
        ax.plot(range(w, len(vals)+1), smoothed, color=color, lw=2)
    ax.set_xlabel("Step")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/grpo_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# Cell 10: Example Negotiation with Trained Model
# ============================================================
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

sample = dataset[0]
conv, idx = play_negotiation(
    model, tokenizer, sample["prompt"], sample["prompt_2"],
    agent_starts=True, max_rounds=CONFIG["max_rounds"],
    temperature=0.7,
)

print("=" * 60)
print("Example negotiation (trained model)")
print("=" * 60)

for i, msg in enumerate(conv):
    marker = " (AGENT)" if i in idx else " (OPPONENT)"
    role = "🏠 Landlord" if msg["role"] == "assistant" else "🧑 Tenant"
    print(f"\n{role}{marker}:")
    print(f"  {msg['content']}")

# Evaluate with Luca's reward function
reward = compute_rewards(reward_functions, sample, [conv])
print(f"\n--- Reward: {reward[0]:.0f} / 100 ---")


In [ ]:
# ============================================================
# Cell 11: Save Final Model
# ============================================================
model.save_pretrained("/content/grpo_final")
tokenizer.save_pretrained("/content/grpo_final")
print("Model saved to /content/grpo_final")

# Optional: Upload to HuggingFace
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
# model.push_to_hub("your-username/grpo-negotiator")
# tokenizer.push_to_hub("your-username/grpo-negotiator")
